In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim

from torch.nn.utils.fusion import fuse_conv_bn_eval

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import onnx

import matplotlib.pyplot as plt
import numpy as np

from datetime import datetime
from time import time
import copy

In [2]:
import json_to_pytorch

In [3]:
def get_loaders(batch_size=64, num_workers=2, normalize=False):
    # Statistiche CIFAR-10 standard
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2470, 0.2435, 0.2616)

    if normalize:
      train_tf = transforms.Compose([
          transforms.RandomCrop(32, padding=4),       # augmentation fondamentale
          transforms.RandomHorizontalFlip(),
          transforms.ToTensor(),
          transforms.Normalize(mean, std),
      ])
      test_tf = transforms.Compose([
          transforms.ToTensor(),
          transforms.Normalize(mean, std),
      ])
    else:
      train_tf = transforms.Compose([
          transforms.RandomCrop(32, padding=4),       # augmentation fondamentale
          transforms.RandomHorizontalFlip(),
          transforms.ToTensor(),
      ])
      test_tf = transforms.Compose([
          transforms.ToTensor(),
      ])

    train_ds = datasets.CIFAR10(root="./data", train=True,  download=True, transform=train_tf)
    test_ds  = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader

In [4]:
criterion = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total

In [5]:
model = json_to_pytorch.json_to_pytorch("../VeryDiffPolyExperiments/results/gelu/best_model_bn_8_0.0001l1_no_pad_50.json", double_precision=True)

In [6]:
model

Sequential(
  (0): FrozenBatchNorm()
  (1): Conv2d(3, 8, kernel_size=(3, 3), stride=(2, 2))
  (2): ChebyshevPoly()
  (3): Conv2d(8, 16, kernel_size=(3, 3), stride=(2, 2))
  (4): ChebyshevPoly()
  (5): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2))
  (6): ChebyshevPoly()
  (7): Flatten(start_dim=1, end_dim=-1)
  (8): Linear(in_features=288, out_features=256, bias=True)
  (9): ChebyshevPoly()
  (10): Linear(in_features=256, out_features=10, bias=True)
)

In [7]:
train_loader, test_loader = get_loaders(batch_size=128)
test_loss,  test_acc  = evaluate(model, test_loader, criterion, "cpu")

Files already downloaded and verified
Files already downloaded and verified


In [8]:
test_acc

0.3006

In [20]:
# Get sample from test_loader
test_input, test_label = next(iter(test_loader))

In [21]:
test_input = test_input[0:1,:,:,:]

In [22]:
import attack

In [23]:
adv = attack.Adversary()
adv_example, true_label, adv_label = adv.generate(model, test_input, 2.0/255.0, attack_iters=20, attack_alpha=1.0, attack_restarts=20)

In [24]:
print(f"True label: {true_label}, Adversarial label: {adv_label}")

True label: 2, Adversarial label: 5


In [25]:
model(test_input)

tensor([[-2.8050,  1.4884,  3.9896,  2.5115, -2.9466,  2.5356, -1.3176, -2.6540,
          1.2762, -1.0730]], dtype=torch.float64, grad_fn=<AddmmBackward0>)

In [26]:
model(adv_example)

tensor([[-4.0724,  0.8627,  2.9431,  2.9050, -3.0144,  4.3708, -1.4860, -1.6617,
          0.7796, -0.9551]], dtype=torch.float64, grad_fn=<AddmmBackward0>)